In [3]:
#Open API Fine Tuning Dashboard URL: https://platform.openai.com/finetune/

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

In [5]:
with open("data/training_data.jsonl", "rb") as f:
    training_file_obj = client.files.create(file=f, purpose="fine-tune")

print("Uploaded training file ID:", training_file_obj.id)

with open("data/validation_data.jsonl", "rb") as f:
    validation_file_obj = client.files.create(file=f, purpose="fine-tune")

print("Uploaded validation file ID:", validation_file_obj.id)

Uploaded training file ID: file-Dbzit2SLAAWgTR6ZngzAAx
Uploaded validation file ID: file-P5xccczMqpYRsb16VwkBkE


In [6]:
# Create a fine-tuning job
job = client.fine_tuning.jobs.create(
    training_file=training_file_obj.id,       # The file ID you uploaded
    validation_file=validation_file_obj.id,     # Optional
    model="gpt-4.1-nano-2025-04-14",               # or "gpt-3.5-turbo", etc.
    suffix="brand-customer-support"       # Optional model name suffix
)

print("Fine-tune job created:", job.id)

Fine-tune job created: ftjob-O87EU57k0Q9SqTD4vNWUZR2Q


In [7]:
jobs = client.fine_tuning.jobs.list(limit=5)
for j in jobs.data:
    print(j.id, j.status, j.fine_tuned_model)

ftjob-O87EU57k0Q9SqTD4vNWUZR2Q validating_files None
ftjob-74KVP2IpbvAk7vcFGeBydlIt succeeded ft:gpt-4.1-nano-2025-04-14:personal:brand-customer-support:D4We9LFA
ftjob-KlxiJb4AJG1CGKJaePC2gDKR succeeded ft:gpt-4.1-nano-2025-04-14:personal:brand-customer-support:D4WNcvBG


In [8]:
def ask_question_without_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

def ask_question_with_finetuning(prompt):
    print(f"User asked: {prompt}")
    response = client.chat.completions.create(
        model="ft:gpt-4.1-nano-2025-04-14:personal:brand-customer-support:D4We9LFA",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
            {"role": "user", "content": prompt}
        ]
    )
    print (response)
    return response.choices[0].message.content  

In [ ]:
import time
while True:
    # Ask user for a question
    user_prompt = input("Ask something: ")

    if (user_prompt.lower() != 'quit'):
        # Get and print the response
        response = ask_question_without_finetuning(user_prompt)
        print("\n[WITHOUT FINE TUNING] OpenAI says:", response)

        response = ask_question_with_finetuning(user_prompt)
        print("\n[WITH FINE TUNING] OpenAI says:", response)

        # add delay of 3 seconds
        time.sleep(3)
    else:
        break    

User asked: My coffee machine arrived broken and I can’t get anyone on the phone.
ChatCompletion(id='chatcmpl-D6mNyOPqRbHHhHX0WHzVAnEkngfiu', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Sorry that happened. I can help you get this sorted even without a phone call. Here’s a quick plan.\n\nWhat to do now\n- Document it: take clear photos of the broken machine and the packaging, and save the box and contents.\n- Find your order details: order number, seller/brand, purchase date, and payment method.\n- Try an alternative support channel:\n  - Email or online chat on the retailer/manufacturer’s site\n  - Social media direct message (Facebook/X/Twitter)\n  - Return/exchange portal or ticket system in your account\n  - If you used a marketplace (Amazon/Walmart/etc.), use their returns/damage claim process\n- If it was damaged in transit, file a carrier damage claim as well (often you can do this via the carrier’s website or the retailer